# Check citation faithfulness in RAG with a zero-token gate

When a RAG answer cites its sources, the citations can fail in ways that read as
completely authoritative — and an LLM asked "does this quote support the claim?"
waves them through, because they *look* fluent and supportive:

- **fabricated** — a quoted span that appears in no retrieved document;
- **frankenquote** — every word is real, but the exact span was never written
  contiguously in the source;
- **misattributed** — a real span, but attributed to the wrong document.

This cookbook shows a *cheap deterministic detector → expensive judge* pattern for
catching these before the answer reaches a user:

1. Ask the model, via **Structured Outputs**, to return each claim with the
   `document_id` it relies on and a short **verbatim quote** from that document.
2. Run a **0-token verbatim gate** (pure Python — no model, no API key) that checks
   each quote really appears in the cited document. This alone rejects the three
   failure modes above and runs offline.
3. Only for quotes that pass the gate, optionally call a **burden-of-proof judge**
   to decide whether the quote actually *supports* the claim (a right quote can
   still be the wrong evidence). Fabrications never reach the judge, so they cost
   zero tokens.

The gate is inlined here; its standalone, framework-agnostic version (gate +
burden-of-proof judge) lives at
[`verbatim-citation-gate`](https://github.com/Palo-Alto-AI-Research-Lab/verbatim-citation-gate).

## The verbatim gate (deterministic, runs offline)

In [ ]:
import re


def normalize(text: str) -> str:
    """Case/typography/whitespace-insensitive form for verbatim matching."""
    text = text.lower()
    text = re.sub(r"[‘’]", "'", text)
    text = re.sub(r"[“”]", '"', text)
    text = re.sub(r"[–—]", "-", text)
    text = re.sub(r"[^a-z0-9%.]+", " ", text)
    return " ".join(text.split())


def gate(quote: str, cited_doc_id: str, docs: dict) -> str:
    """Return 'found' | 'misattributed' | 'not_found'. Fails closed on empty quotes."""
    q = normalize(quote)
    if not q:
        return "not_found"
    cited = docs.get(cited_doc_id)
    if cited is not None and q in normalize(cited):
        return "found"
    if any(q in normalize(t) for d, t in docs.items() if d != cited_doc_id):
        return "misattributed"
    return "not_found"

### Verify it offline on structured citations

No API key needed here — this is the shape of citations Structured Outputs returns,
with one faithful citation and the three planted failure modes.

In [ ]:
DOCS = {
    "doc_0": "GPT-4o has a context window of 128,000 tokens.",
    "doc_1": "text-embedding-3-large produces embeddings with 3,072 dimensions.",
}

CITATIONS = [
    {"claim": "GPT-4o supports a 128k context.", "document_id": "doc_0",
     "quote": "context window of 128,000 tokens"},                          # faithful
    {"claim": "The large embedding model outputs 3072 dims.", "document_id": "doc_0",
     "quote": "embeddings with 3,072 dimensions"},                          # wrong doc
    {"claim": "GPT-4o outputs 3072-dim embeddings.", "document_id": "doc_1",
     "quote": "GPT-4o produces 3,072 dimensions"},                          # frankenquote
    {"claim": "GPT-4o has a 1M token context.", "document_id": "doc_0",
     "quote": "context window of 1,000,000 tokens"},                        # fabricated
]

for c in CITATIONS:
    status = gate(c["quote"], c["document_id"], DOCS)
    flag = "OK  " if status == "found" else "FLAG"
    print(f"{flag} [{status:>13}]  {c['claim']}")

`found` citations are safe to surface; `misattributed` and `not_found` should be
flagged or dropped — all decided deterministically, for zero tokens.

## Generate the citations with Structured Outputs

With an API key, ask the model to answer **and** return structured citations, then
run the same gate over them. Uses the Responses API with a Pydantic schema; needs
`OPENAI_API_KEY` (not run in CI).

In [ ]:
# pip install openai pydantic
import os

if not os.getenv("OPENAI_API_KEY"):
    print("Set OPENAI_API_KEY to run the live example.")
else:
    from openai import OpenAI
    from pydantic import BaseModel

    class Citation(BaseModel):
        claim: str
        document_id: str
        quote: str

    class CitedAnswer(BaseModel):
        answer: str
        citations: list[Citation]

    client = OpenAI()
    library = "\n".join(f"[{doc_id}] {text}" for doc_id, text in DOCS.items())
    resp = client.responses.parse(
        model="gpt-4.1",
        input=[
            {"role": "system", "content": "Answer using only the library. For each claim, cite the document_id "
             "and a short quote copied verbatim from that document."},
            {"role": "user", "content": f"Library:\n{library}\n\nQuestion: What context window does GPT-4o have, "
             "and how many dimensions does text-embedding-3-large output?"},
        ],
        text_format=CitedAnswer,
    )
    cited = resp.output_parsed
    print(cited.answer, "\n")
    for c in cited.citations:
        status = gate(c.quote, c.document_id, DOCS)
        flag = "OK  " if status == "found" else "FLAG"
        print(f"{flag} [{status:>13}]  {c.quote!r} -> {c.document_id}")

## Optional: a burden-of-proof judge for the ambiguous case

The gate settles whether a quote *exists*. Whether a real, correctly-attributed
quote actually **supports** its claim is a judgment call best given to a model — but
with the burden of proof on the citation: the verdict defaults to *unsupported*, the
model's outside knowledge is inadmissible, and an unparseable verdict fails closed.
Because only `found` quotes reach it, fabricated and misattributed citations cost
zero judge calls.

A ready-made, model-agnostic version of this judge is in
[`verbatim-citation-gate`](https://github.com/Palo-Alto-AI-Research-Lab/verbatim-citation-gate);
plug your `client.responses.create` call into its `llm_call` hook.